In [3]:
import pandas as pd
import pandera.pandas as pa
import json 
import logging 

In [52]:
# Configure logging
logging.basicConfig(
    filename="validation_errors.log",
    filemode="w",
    format="%(asctime)s - %(message)s",
    level=logging.INFO,
)

In [6]:
valid_data = pd.DataFrame({
    "class": ["Benign", "Benign", "Malignant"],
    "mean_radius": [6.0, 31.2, 22.8]
})

invalid_data = pd.DataFrame({
    "class": ["Benign", "Benign", "benign", "Malignant"],
    "mean_radius": [6.0, 6.0, 31.2, -9999]
})

In [50]:
schema = pa.DataFrameSchema(
    {
        "class": pa.Column(str, pa.Check.isin(["Benign", "Malignant"])),
        "mean_radius": pa.Column(float, pa.Check.between(5, 54), nullable=True)
    },
    checks=[
        pa.Check(lambda df: ~df.duplicated(), error="Duplicate rows found"),
        pa.Check(lambda df: ~(df.isna().all(axis=1)), error="Empty rows found.")
    ]
)

In [53]:
# Initialize error cases DataFrame
error_cases = pd.DataFrame()
data = invalid_data.copy()

# Validate data and handle errors
try:
    validated_data = schema.validate(data, lazy=True)
except pa.errors.SchemaErrors as e:
    error_cases = e.failure_cases

    # Convert the error message to a JSON string
    error_message = json.dumps(e.message, indent=2)
    logging.error("\n" + error_message)

# Filter out invalid rows based on the error cases
if not error_cases.empty:
    invalid_indices = error_cases["index"].dropna().unique()
    validated_data = (
        data.drop(index=invalid_indices)
        .reset_index(drop=True)
        .drop_duplicates()
        .dropna(how="all")
    )
else:
    validated_data = data